# RealMLP Model — PharmShed Super Dataset

**Model:** RealMLP-TD (pytabkit)  
**Task:** Multi-class classification — predict which of 216 pharmaceuticals a person is prescribed  
**Features:** Demographics (Age, Sex, Family_income, Insurance_coverage, Race_ethnicity) + Prescription (Quantity, Form, Strength, Day_Supply)  
**Split strategy:** StratifiedGroupKFold (5-fold CV), grouped by Person_ID to prevent data leakage  
**Missing values:** Prescription features are NaN for no-prescription records — filled with -1 as sentinel value before model training  
**Metrics:** Cohen's Kappa, MCC, macro/micro averaged precision, recall, F2 score  

In [ ]:
# Install required libraries.
# pytabkit provides RealMLP-TD, permetrics provides macro/micro evaluation metrics.
!pip install pytabkit permetrics

In [ ]:
# Import all required libraries.
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score, matthews_corrcoef,
    classification_report
)
from pytabkit import RealMLP_TD_Classifier
from permetrics import ClassificationMetric
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load the super integrated dataset (2014-2021) and metadata.
# The super dataset contains demographics + prescription features joined on Observation_ID.
# Metadata provides Person_ID which is used only for StratifiedGroupKFold grouping.
super_df = pd.read_csv('super_integrated_data.csv')
metadata = pd.read_csv('metadata.csv')

if 'Unnamed: 0' in super_df.columns:
    super_df = super_df.drop(columns=['Unnamed: 0'])
if 'Unnamed: 0' in metadata.columns:
    metadata = metadata.drop(columns=['Unnamed: 0'])

print('Super dataset shape:', super_df.shape)
print('Metadata shape:', metadata.shape)
print('\nSuper dataset columns:', super_df.columns.tolist())
print('\nMissing values:')
print(super_df.isnull().sum())

In [ ]:
# Join Person_ID from metadata onto the super dataset.
# Person_ID is not a model feature — it is used only to group records by person
# in StratifiedGroupKFold so that all rows for the same person stay in the same fold.
# This prevents data leakage from the same person appearing in both train and validation.
person_id_map = metadata[['Observation_ID', 'Person_ID']]
super_df = super_df.merge(person_id_map, on='Observation_ID', how='left')

print('Columns after join:', super_df.columns.tolist())
print('Shape after join:', super_df.shape)
print('Missing Person_IDs:', super_df['Person_ID'].isnull().sum())

In [ ]:
# Handle missing values in prescription features.
# Records with Drug = 'no prescriptions' have NaN for all prescription features
# because no prescription was dispensed. RealMLP does not handle NaN natively,
# so these are filled with -1 as a sentinel value.
# The model learns that -1 signals absence of prescription data,
# which is itself a meaningful pattern for the no-prescription class.
prescription_cols = ['Quantity', 'Form', 'Strength', 'Day_Supply']
super_df[prescription_cols] = super_df[prescription_cols].fillna(-1)

print('Missing values after filling with -1:')
print(super_df.isnull().sum())

In [ ]:
# Exploratory check before modeling.
# Verify class count, drug distribution, and person count.
print('Unique persons:', super_df['Person_ID'].nunique())
print('Unique drugs:', super_df['Drug'].nunique())
print('No prescription rows:', (super_df['Drug'] == 'no prescriptions').sum())
print('Actual drug rows:', (super_df['Drug'] != 'no prescriptions').sum())

print('\nTop 10 most prescribed drugs:')
print(super_df['Drug'].value_counts().head(10))

print('\nBottom 5 rarest drugs:')
print(super_df['Drug'].value_counts().tail(5))

In [ ]:
# Define feature columns, categorical columns, and target.
# Form is categorical because it contains drug form codes (TABS, CAPS, ORAL, etc.).
# Quantity, Strength, Day_Supply are numeric — RealMLP scales these internally.
# RealMLP handles all encoding and scaling internally — do not preprocess further.
feature_cols     = ['Age', 'Sex', 'Family_income', 'Insurance_coverage',
                    'Race_ethnicity', 'Quantity', 'Form', 'Strength', 'Day_Supply']
categorical_cols = ['Sex', 'Insurance_coverage', 'Race_ethnicity', 'Form']
numeric_cols     = ['Age', 'Family_income', 'Quantity', 'Strength', 'Day_Supply']
target_col       = 'Drug'

# Fit LabelEncoder once on the full dataset before the CV loop.
# Fitting inside the loop would produce different integer mappings per fold,
# making fold results incomparable and breaking per-drug recall aggregation.
le = LabelEncoder()
super_df['Drug_encoded'] = le.fit_transform(super_df[target_col])

print('Unique classes:', len(le.classes_))
print('Feature cols:', feature_cols)
print('Categorical cols:', categorical_cols)
print('\nSample drug to integer mapping (first 5):')
for i, drug in enumerate(le.classes_[:5]):
    print(f'  {drug} -> {i}')

In [ ]:
# 5-fold stratified group cross-validation.
#
# StratifiedGroupKFold ensures:
#   1. Each fold has similar drug class distribution (stratified)
#   2. All records for the same person stay in the same fold (grouped)
#      preventing the model from seeing the same person in both train and val
#
# For each fold:
#   1. Split by Person_ID groups and drug label stratification
#   2. Shuffle rows within train and val (prevents refill records clustering together)
#   3. Set categorical dtypes so RealMLP uses its built-in categorical encoding
#   4. Train RealMLP-TD on train fold
#   5. Predict on val fold
#   6. Compute and store all metrics

sgkf = StratifiedGroupKFold(n_splits=5)

X      = super_df[feature_cols].copy()
y      = super_df['Drug_encoded'].values
groups = super_df['Person_ID'].values

fold_results          = []
per_drug_recall_folds = []

for fold_num, (train_idx, val_idx) in enumerate(sgkf.split(X, y, groups), start=1):
    print(f'\n{"="*50}')
    print(f'FOLD {fold_num}/5')
    print(f'{"="*50}')

    X_train_fold = X.iloc[train_idx].copy()
    X_val_fold   = X.iloc[val_idx].copy()
    y_train_fold = y[train_idx]
    y_val_fold   = y[val_idx]

    # Shuffle within each fold to prevent refill records from clustering.
    train_order  = np.random.RandomState(42).permutation(len(X_train_fold))
    val_order    = np.random.RandomState(42).permutation(len(X_val_fold))
    X_train_fold = X_train_fold.iloc[train_order].reset_index(drop=True)
    y_train_fold = y_train_fold[train_order]
    X_val_fold   = X_val_fold.iloc[val_order].reset_index(drop=True)
    y_val_fold   = y_val_fold[val_order]

    # Set categorical dtypes so RealMLP applies its built-in categorical encoding.
    # Do not one-hot encode manually — RealMLP handles this internally.
    for col in categorical_cols:
        X_train_fold[col] = X_train_fold[col].astype('category')
        X_val_fold[col]   = X_val_fold[col].astype('category')

    print(f'Train size: {len(X_train_fold):,} | Val size: {len(X_val_fold):,}')
    print(f'Unique drugs in train: {len(np.unique(y_train_fold))} | val: {len(np.unique(y_val_fold))}')

    # Train RealMLP-TD.
    # TD version uses tuned default hyperparameters — no manual search needed.
    # Change device to 'cuda' for NVIDIA GPU or 'cpu' if no GPU available.
    model = RealMLP_TD_Classifier(
        device='mps',
        random_state=42,
        n_epochs=30,
    )
    model.fit(X_train_fold, y_train_fold)
    print(f'Fold {fold_num} training complete.')

    y_pred_fold = model.predict(X_val_fold)

    # Overall metrics.
    acc   = accuracy_score(y_val_fold, y_pred_fold)
    kappa = cohen_kappa_score(y_val_fold, y_pred_fold)
    mcc   = matthews_corrcoef(y_val_fold, y_pred_fold)

    # Macro and micro averaged metrics via permetrics.
    # Macro: average per class equally — treats rare and common drugs equally.
    # Micro: aggregate all counts — dominated by the most common drugs.
    evaluator = ClassificationMetric(y_val_fold, y_pred_fold)

    macro_precision = evaluator.precision_score(average='macro')
    micro_precision = evaluator.precision_score(average='micro')
    macro_recall    = evaluator.recall_score(average='macro')
    micro_recall    = evaluator.recall_score(average='micro')
    macro_f1        = evaluator.f1_score(average='macro')
    micro_f1        = evaluator.f1_score(average='micro')

    # F2 score weights recall twice as much as precision.
    # Higher recall is more important here because missing a drug class
    # means underestimating its wastewater load.
    macro_f2 = evaluator.fbeta_score(beta=2, average='macro')
    micro_f2 = evaluator.fbeta_score(beta=2, average='micro')

    fold_results.append({
        'fold': fold_num,
        'accuracy': acc,
        'cohen_kappa': kappa,
        'mcc': mcc,
        'macro_precision': macro_precision,
        'micro_precision': micro_precision,
        'macro_recall': macro_recall,
        'micro_recall': micro_recall,
        'macro_f1': macro_f1,
        'micro_f1': micro_f1,
        'macro_f2': macro_f2,
        'micro_f2': micro_f2,
    })

    # Per-drug recall for this fold.
    # Used for ensemble model selection — the ensemble picks the best model per drug.
    report = classification_report(
        y_val_fold, y_pred_fold,
        labels=np.arange(len(le.classes_)),
        target_names=le.classes_,
        output_dict=True,
        zero_division=0
    )
    drug_recalls         = {drug: report[drug]['recall'] for drug in le.classes_ if drug in report}
    drug_recalls['fold'] = fold_num
    per_drug_recall_folds.append(drug_recalls)

    print(f'Fold {fold_num} results:')
    print(f'  Accuracy:      {acc:.4f}')
    print(f'  Cohen Kappa:   {kappa:.4f}')
    print(f'  MCC:           {mcc:.4f}')
    print(f'  Macro Recall:  {macro_recall:.4f}')
    print(f'  Micro Recall:  {micro_recall:.4f}')
    print(f'  Macro F2:      {macro_f2:.4f}')

print('\n' + '='*50)
print('ALL FOLDS COMPLETE')
print('='*50)

In [ ]:
# Summarize cross-validation results across all 5 folds.
# Report mean and standard deviation for each metric.
# Standard deviation shows how stable the model is across different data splits.
results_df = pd.DataFrame(fold_results)

print('CV RESULTS — MEAN +/- STD ACROSS 5 FOLDS')
print('='*55)
metric_cols = [c for c in results_df.columns if c != 'fold']
for col in metric_cols:
    mean = results_df[col].mean()
    std  = results_df[col].std()
    print(f'  {col:<25}: {mean:.4f} +/- {std:.4f}')

results_df.to_csv('realmlp_super_cv_results.csv', index=False)
print('\nCV results saved to realmlp_super_cv_results.csv')

In [ ]:
# Compute average per-drug recall across all 5 folds.
# This is the key output for ensemble model selection.
# The ensemble picks whichever base model has the highest recall for each drug.
per_drug_df = pd.DataFrame(per_drug_recall_folds)
drug_cols   = [c for c in per_drug_df.columns if c != 'fold']

mean_drug_recall = per_drug_df[drug_cols].mean().reset_index()
mean_drug_recall.columns = ['Drug', 'Mean_Recall_RealMLP_Super']
mean_drug_recall = mean_drug_recall.sort_values('Mean_Recall_RealMLP_Super', ascending=False)

print('Top 20 drugs by mean recall:')
print(mean_drug_recall.head(20).to_string(index=False))

print('\nBottom 20 drugs by mean recall:')
print(mean_drug_recall.tail(20).to_string(index=False))

mean_drug_recall.to_csv('realmlp_super_per_drug_recall.csv', index=False)
print('\nPer-drug recall saved to realmlp_super_per_drug_recall.csv')

In [ ]:
# Train final model on the full training dataset (2014-2021).
# After CV gives confidence in model performance, retrain on all available data
# to maximize signal before evaluating on the held-out 2022 validation set.
X_final = super_df[feature_cols].copy()
y_final = super_df['Drug_encoded'].values

for col in categorical_cols:
    X_final[col] = X_final[col].astype('category')

X_final, y_final = shuffle(X_final, y_final, random_state=42)
X_final = X_final.reset_index(drop=True)

print('Final training data size:', X_final.shape)
print('Number of classes:', len(le.classes_))
print('\nTraining final model...')

final_model = RealMLP_TD_Classifier(
    device='mps',
    random_state=42,
    n_epochs=30,
)
final_model.fit(X_final, y_final)
print('Final model training complete.')

In [ ]:
# Internal validation on held-out MEPS 2022 data.
# The 2022 dataset was never seen during training or CV.
# The same super dataset construction and preprocessing applied to training
# must be applied identically to the 2022 data before evaluation.

# Load 2022 super dataset (should be built with same join and preprocessing as training)
data_2022 = pd.read_csv('super_data_2022.csv')
if 'Unnamed: 0' in data_2022.columns:
    data_2022 = data_2022.drop(columns=['Unnamed: 0'])

# Fill prescription NaNs with -1 (same as training)
data_2022[prescription_cols] = data_2022[prescription_cols].fillna(-1)

print('2022 data shape:', data_2022.shape)

# Drop any drugs in 2022 that were not seen during training
unseen = set(data_2022['Drug'].unique()) - set(le.classes_)
print(f'Unseen drugs in 2022 (will be dropped): {unseen}')
data_2022 = data_2022[data_2022['Drug'].isin(le.classes_)].reset_index(drop=True)
print(f'2022 rows after filtering: {len(data_2022):,}')

X_2022 = data_2022[feature_cols].copy()
for col in categorical_cols:
    X_2022[col] = X_2022[col].astype('category')

y_2022_encoded = le.transform(data_2022['Drug'])
y_pred_2022    = final_model.predict(X_2022)

# Compute validation metrics
acc_2022   = accuracy_score(y_2022_encoded, y_pred_2022)
kappa_2022 = cohen_kappa_score(y_2022_encoded, y_pred_2022)
mcc_2022   = matthews_corrcoef(y_2022_encoded, y_pred_2022)

evaluator_2022   = ClassificationMetric(y_2022_encoded, y_pred_2022)
macro_prec_2022  = evaluator_2022.precision_score(average='macro')
micro_prec_2022  = evaluator_2022.precision_score(average='micro')
macro_recall_2022 = evaluator_2022.recall_score(average='macro')
micro_recall_2022 = evaluator_2022.recall_score(average='micro')
macro_f2_2022    = evaluator_2022.fbeta_score(beta=2, average='macro')
micro_f2_2022    = evaluator_2022.fbeta_score(beta=2, average='micro')

print('\nINTERNAL VALIDATION — MEPS 2022 Results')
print('='*50)
print(f'Accuracy:          {acc_2022:.4f}')
print(f'Cohen Kappa:       {kappa_2022:.4f}')
print(f'MCC:               {mcc_2022:.4f}')
print(f'Macro Precision:   {macro_prec_2022:.4f}')
print(f'Micro Precision:   {micro_prec_2022:.4f}')
print(f'Macro Recall:      {macro_recall_2022:.4f}')
print(f'Micro Recall:      {micro_recall_2022:.4f}')
print(f'Macro F2:          {macro_f2_2022:.4f}')
print(f'Micro F2:          {micro_f2_2022:.4f}')

In [ ]:
# Save per-drug metrics and overall validation summary for 2022.
report_2022 = classification_report(
    y_2022_encoded, y_pred_2022,
    labels=np.arange(len(le.classes_)),
    target_names=le.classes_,
    output_dict=True,
    zero_division=0
)

drug_recall_2022 = pd.DataFrame([
    {
        'Drug': drug,
        'Recall_2022': report_2022[drug]['recall'],
        'Precision_2022': report_2022[drug]['precision'],
        'F1_2022': report_2022[drug]['f1-score'],
        'Support_2022': report_2022[drug]['support']
    }
    for drug in le.classes_ if drug in report_2022
]).sort_values('Recall_2022', ascending=False)

print('Top 15 drugs by recall on 2022 data:')
print(drug_recall_2022.head(15).to_string(index=False))
print('\nBottom 15 drugs by recall on 2022 data:')
print(drug_recall_2022.tail(15).to_string(index=False))

drug_recall_2022.to_csv('realmlp_super_2022_per_drug_metrics.csv', index=False)
print('\nPer-drug 2022 metrics saved to realmlp_super_2022_per_drug_metrics.csv')

validation_summary = pd.DataFrame([{
    'model': 'RealMLP_Super',
    'dataset': 'MEPS_2022_internal_validation',
    'accuracy': acc_2022,
    'cohen_kappa': kappa_2022,
    'mcc': mcc_2022,
    'macro_precision': macro_prec_2022,
    'micro_precision': micro_prec_2022,
    'macro_recall': macro_recall_2022,
    'micro_recall': micro_recall_2022,
    'macro_f2': macro_f2_2022,
    'micro_f2': micro_f2_2022,
}])
validation_summary.to_csv('realmlp_super_validation_summary.csv', index=False)
print('Validation summary saved to realmlp_super_validation_summary.csv')

## Output Files

| File | Contents |
|------|----------|
| `realmlp_super_cv_results.csv` | Mean +/- std for all metrics across 5 CV folds |
| `realmlp_super_per_drug_recall.csv` | Average recall per drug across 5 folds (for ensemble selection) |
| `realmlp_super_2022_per_drug_metrics.csv` | Per-drug recall, precision, F1 on MEPS 2022 internal validation |
| `realmlp_super_validation_summary.csv` | Overall validation metrics on MEPS 2022 |

**Next step:** Compare these results against the demographics-only RealMLP baseline to assess whether the prescription features improve model performance.